<a href="https://colab.research.google.com/github/zk025815-beep/HCCDA-AI-Course/blob/main/AI_Powered_Lead_Generation_%26_Classification_basic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Business Leads Classifier

### Overview
Extracts local business leads from the **Geoapify Places API** and dynamically classifies them as **HOT** or **COLD** prospects using **Gemini**.

### Process Flow
1. **Extraction**: Fetch 10 local restaurant leads via Geoapify API.
2. **Parsing**: Structure key details (name, contact, website, category).
3. **Classification**: Evaluate and label leads using the `gemini-3.5-flash-lite` model.
4. **Export**: Save the finalized classification results into a Pandas DataFrame and export to a CSV file.

In [ ]:
!pip install requests google-genai

In [ ]:
!pip install -q google-genai

In [ ]:
import requests
from google.colab import userdata

GEOAPIFY_API_KEY = userdata.get("GEOAPIFY_API_KEY")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("Geoapify key loaded:", GEOAPIFY_API_KEY is not None)
print("Gemini key loaded:", GEMINI_API_KEY is not None)

In [ ]:
url = "https://api.geoapify.com/v2/places"

params = {
    "categories": "catering.restaurant",
    "filter": "circle:74.3587,31.5204,10000",
    "limit": 10,
    "apiKey": GEOAPIFY_API_KEY
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

In [ ]:
data = response.json()

print("Number of leads:", len(data["features"]))

In [ ]:
import json

print(json.dumps(data["features"][0], indent=2))

In [ ]:
def extract_leads(data):
    leads = []

    for feature in data["features"]:
        properties = feature["properties"]

        categories = properties.get("categories", [])

        lead = {
            "name": properties.get("name", "Unknown"),
            "website": properties.get("website", "Not available"),
            "location": properties.get("formatted", "Not available"),
            "description": properties.get("description", "Not available"),
            "categories": ", ".join(categories)
        }

        leads.append(lead)

    return leads

In [ ]:
leads = extract_leads(data)
print("Total leads:", len(leads))

In [ ]:
leads = extract_leads(data)

for i, lead in enumerate(leads, start=1):
    print(f"\nLead {i}")
    print("Name:", lead["name"])
    print("Website:", lead["website"])
    print("Location:", lead["location"])
    print("Description:", lead["description"])
    print("Categories:", lead["categories"])

In [ ]:
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

In [ ]:
# Test for One Lead First
lead = leads[0]

prompt = f"""
You are an AI sales lead classifier.

Classify the following business lead as either HOT or COLD.

HOT means the business appears to be a strong potential customer
with an established commercial presence.

COLD means the business appears to have lower potential based
on the available information.

Business Name: {lead["name"]}
Website: {lead["website"]}
Location: {lead["location"]}
Description: {lead["description"]}
Categories: {lead["categories"]}

Return your answer in this exact format:

Classification: HOT or COLD
Reason: One short sentence explaining the decision.
"""

In [ ]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)

In [ ]:
#Created a function so that we can Reuse it
def classify_lead(lead):
    prompt = f"""
You are an AI sales lead classifier.

Classify the following business lead as either HOT or COLD.

HOT means the business appears to be a strong potential customer
with an established commercial presence.

COLD means the business appears to have lower potential based
on the available information.

Business Name: {lead["name"]}
Website: {lead["website"]}
Location: {lead["location"]}
Description: {lead["description"]}
Categories: {lead["categories"]}

Return your answer in exactly this format:

Classification: HOT or COLD
Reason: One short sentence explaining the decision.
"""

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return response.text

In [ ]:
result = classify_lead(leads[5])

print(result)

result2 = classify_lead(leads[9])

print(result2)

In [ ]:
results = []

for lead in leads:
    classification = classify_lead(lead)

    results.append({
        "name": lead["name"],
        "website": lead["website"],
        "location": lead["location"],
        "description": lead["description"],
        "classification": classification
    })

print("Finished classifying", len(results), "leads.")

In [ ]:
for i, result in enumerate(results, start=1):
    print("=" * 60)
    print(f"LEAD {i}")
    print("=" * 60)

    print("Business:", result["name"])
    print("Website:", result["website"])
    print("Location:", result["location"])
    print("Description:", result["description"])
    print(result["classification"])
    print()

In [ ]:
import pandas as pd

# Convert results list to a DataFrame
df_results = pd.DataFrame(results)

# Display the DataFrame
display(df_results)

In [ ]:
# Save the DataFrame to a CSV file
df_results.to_csv('classified_leads.csv', index=False)
print("DataFrame successfully saved to 'classified_leads.csv'!")